**1. Setup**

In [ ]:
!pip install -U ultralytics remotezip
import ultralytics
ultralytics.checks()

**2. Configure & mount to Google Drive**

Gestures used: dislike, fist, like, stop, one.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

CLASSES = ["dislike", "fist", "like", "one", "stop",  "no_gesture"]
TRAIN_PER_CLASS = 300
VAL_PER_CLASS = 60
IMG_SIZE = 640

WORK = Path("/content/hagrid_yolo")
SAVE = Path("/content/drive/MyDrive/hgr-game-control")
WORK.mkdir(parents=True, exist_ok=True)
SAVE.mkdir(parents=True, exist_ok=True)

**3. Annotate dataset**

Gesture classes pulled from [HaGRID GitHub repo](https://github.com/hukenovs/hagrid).

In [ ]:
!wget -q -O /content/annotations.zip "https://rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru/datasets/hagrid_v2/annotations_with_landmarks/annotations.zip"
!unzip -q -o /content/annotations.zip -d /content/hagrid_ann

import json
ann_root = Path("/content/hagrid_ann")

def find_ann(split, cls):
    hits = list(ann_root.rglob(f"{split}/{cls}.json"))
    assert hits, f"no annotation file for {split}/{cls}"
    return hits[0]

ann = json.load(open(find_ann("train", "like")))
key = next(iter(ann))
print(key)
print({k: v for k, v in ann[key].items() if k != "hand_landmarks"})

**4. Convert labels**

HaGRID gives top-left box corner coordinates (x,y), but YOLO takes center coordinates (cx,cy). Convert cx = x + w/2 and cy = y + h/2.

In [ ]:
import io, random
from PIL import Image
from remotezip import RemoteZip
from tqdm.auto import tqdm

OLD = "https://rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru/datasets/hagrid/hagrid_dataset_new_554800/hagrid_dataset"
V2  = "https://rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru/datasets/hagrid_v2/hagrid_v2_zip"
ZIP_URLS = {
    "dislike": f"{OLD}/dislike.zip", "fist": f"{OLD}/fist.zip",
    "like": f"{OLD}/like.zip",       "one": f"{V2}/one.zip",
    "stop": f"{OLD}/stop.zip",       "no_gesture": f"{V2}/no_gesture.zip"
}
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}

def yolo_lines(entry):    #bbox conversion from corner to center
    lines = []
    for (x, y, w, h), label in zip(entry["bboxes"], entry["labels"]):
        if label not in CLASS_TO_ID:
            continue
        lines.append(f"{CLASS_TO_ID[label]} {x + w/2:.6f} {y + h/2:.6f} {w:.6f} {h:.6f}")
    return lines

import threading
from concurrent.futures import ThreadPoolExecutor

for split in ["train", "val"]:
    (WORK/"images"/split).mkdir(parents=True, exist_ok=True)
    (WORK/"labels"/split).mkdir(parents=True, exist_ok=True)

tls = threading.local()
def get_zip(url):
    if getattr(tls, "url", None) != url:
        tls.z, tls.url = RemoteZip(url), url
    return tls.z

def plan(cls, split, n, seed=0):
    ann = json.load(open(find_ann(split, cls)))
    keys = sorted(ann)
    random.Random(seed).shuffle(keys)
    with RemoteZip(ZIP_URLS[cls]) as z:
        members = {Path(m).stem: m for m in z.namelist()
                   if m.lower().endswith((".jpg", ".jpeg"))}
    items = []
    for key in keys:
        lines = yolo_lines(ann[key])
        if key in members and lines:
            items.append((cls, split, key, members[key], lines))
        if len(items) >= n:
            break
    return items

def fetch_one(item):
    cls, split, key, member, lines = item
    out = WORK/"images"/split/f"{cls}_{key}.jpg"
    if not out.exists():
        img = Image.open(io.BytesIO(get_zip(ZIP_URLS[cls]).read(member))).convert("RGB")
        img.thumbnail((IMG_SIZE, IMG_SIZE))
        tmp = out.with_suffix(".tmp")
        img.save(tmp, "JPEG", quality=90)
        tmp.rename(out)
    (WORK/"labels"/split/f"{cls}_{key}.txt").write_text("\n".join(lines))

for cls in CLASSES:
    items = plan(cls, "train", TRAIN_PER_CLASS) + plan(cls, "val", VAL_PER_CLASS)
    with ThreadPoolExecutor(max_workers=8) as ex:
        list(tqdm(ex.map(fetch_one, items), total=len(items), desc=cls))

In [ ]:
from pathlib import Path

CLASSES = ["dislike", "fist", "like", "one", "stop", "ok", "peace", "no_gesture"]
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}
WORK = Path("/content/hagrid_yolo")
print("dataset folder exists:", WORK.exists())

for split in ["train", "val"]:
    standalone = inside = 0
    ng = f"{CLASS_TO_ID['no_gesture']} "
    for f in (WORK/"labels"/split).glob("*.txt"):
        n = sum(1 for line in f.read_text().splitlines() if line.startswith(ng))
        if f.name.startswith("no_gesture_"):
            standalone += n
        else:
            inside += n
    print(split, "| standalone images:", standalone, "| inside gesture images:", inside)

In [ ]:
print(len(list((WORK/"images"/"train").glob("*.jpg"))), "train images saved")

**5. View data**

Visually verify classifications are correct and boxes are not shifted.

In [ ]:
import cv2, matplotlib.pyplot as plt

def show_samples(split="train", n=6):
    paths = random.sample(sorted((WORK/"images"/split).glob("*.jpg")), n)
    fig, axes = plt.subplots(2, 3, figsize=(12, 6))
    for ax, p in zip(axes.ravel(), paths):
        im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        H, W = im.shape[:2]
        for line in (WORK/"labels"/split/f"{p.stem}.txt").read_text().splitlines():
            c, cx, cy, w, h = line.split()
            cx, cy, w, h = float(cx)*W, float(cy)*H, float(w)*W, float(h)*H
            p1 = (int(cx - w/2), int(cy - h/2))
            p2 = (int(cx + w/2), int(cy + h/2))
            cv2.rectangle(im, p1, p2, (0, 255, 0), 2)
            cv2.putText(im, CLASSES[int(c)], (p1[0], max(p1[1] - 5, 12)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        ax.imshow(im); ax.axis("off")
    plt.show()

show_samples()

**6. Dataset configuration**

In [ ]:
yaml_text = f"path: {WORK}\ntrain: images/train\nval: images/val\nnames:\n"
yaml_text += "".join(f"  {i}: {c}\n" for i, c in enumerate(CLASSES))
(WORK / "data.yaml").write_text(yaml_text)
print(yaml_text)

**7. Training smoke test**

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
results = model.train(
    data=str(WORK / "data.yaml"),
    epochs=2,
    imgsz=640,
    batch=16,
    workers=2,
    project=str(SAVE / "runs"),
    name="smoke_test",
)

**8. Back up dataset to Drive**

In [ ]:
import shutil
shutil.make_archive(str(SAVE / "hagrid_subset"), "zip", WORK)

**9. Real training run**

In [ ]:
model = YOLO("yolo26n.pt")
results = model.train(
    data=str(WORK / "data.yaml"),
    epochs=50,
    patience=10,
    imgsz=640,
    batch=32,
    workers=2,
    project=str(SAVE / "runs"),
    name="gestures_n",
)

**10. Retrieve results**

In [ ]:
from IPython.display import Image as IPImage, display

run = Path(model.trainer.save_dir)
display(IPImage(filename=str(run / "results.png")))
display(IPImage(filename=str(run / "confusion_matrix.png")))

best = YOLO(run / "weights" / "best.pt")
metrics = best.val()
print("mAP50:", metrics.box.map50, " mAP50-95:", metrics.box.map)

**11. Back up dataset**

In [ ]:
import shutil
shutil.make_archive(str(SAVE / "hagrid_subset"), "zip", WORK)
!ls -lh "{SAVE}"